In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to cache file
cache_file = "cache_all_slides.pkl"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import subset_df, subset_df_list

df_HE = subset_df(df_all, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
df_HE = subset_df_list(df_HE, "T_category", "Blood, Bone Marrow and Lymphatic System")
# df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")
df_HE = df_HE[df_HE['M_category'].apply(len) == 1]

In [ ]:
all_filenames = df_HE["filename"].tolist()
print("Number of wsi filenames: ", len(all_filenames))

In [ ]:
from helper_functions import with_tissue_artifact

tissue_error = False
if tissue_error: 
    df_sub = with_tissue_artifact(df_all, cache_file, segmentation_type = "tissue", status="error", version="default")
    all_filenames = df_sub["filename"].tolist()
    print("Number of slides with error during default tissue detection: ", len(all_filenames))

In [ ]:
from tissue_artifact_segmentation import SegmentMany

segmenter = SegmentMany(all_filenames, cache_file, zarr_dir, "tissue", version="default")

In [ ]:
# Visualize Single Slide
import os
from wsidata import open_wsi

path = all_filenames[2]
print(path)

zarr_path = os.path.join(zarr_dir, os.path.basename(path).replace(".mrxs", ".zarr"))
wsi = open_wsi(path, zarr_path)
wsi

In [ ]:
import lazyslide as zs
viewer = zs.pl.WSIViewer(wsi)
viewer.add_image()
viewer.add_contours(key = 'tissue_default')
viewer.show()